In [2]:
class Empty(Exception):
    pass

class ArrayQueue:
    """FIFO queue implementation using a Python list as underlying storage."""
    DEFAULT_CAPACITY = 8  # moderate capacity for all new queues

    def __init__(self):
        """Create an empty queue."""
        self._data = [None] * ArrayQueue.DEFAULT_CAPACITY
        self._size = 0
        self._front = 0

    def __len__(self):
        """Return the number of elements in the queue."""
        return self._size

    def is_empty(self):
        """Return True if the queue is empty."""
        return self._size == 0

    def first(self):
        """Return (but do not remove) the element at the front of the queue.
        Raise Empty exception if the queue is empty.
        """
        if self.is_empty():
            raise Empty('Queue is empty')
        return self._data[self._front]

    def dequeue(self):
        """Remove and return the first element of the queue (i.e., FIFO).
        Raise Empty exception if the queue is empty.
        """
        if self.is_empty():
            raise Empty('Queue is empty')
        answer = self._data[self._front]
        self._data[self._front] = None          # help garbage collection
        self._front = (self._front + 1) % len(self._data)
        self._size -= 1
        return answer

    def enqueue(self, e):
        """Add an element to the back of queue."""
        if self._size == len(self._data):
            self._resize(2 * len(self._data))   # double the array size
        avail = (self._front + self._size) % len(self._data)
        self._data[avail] = e
        self._size += 1

    def _resize(self, cap):
        """Resize to a new list of capacity >= len(self)."""
        old = self._data                         # keep track of existing list
        self._data = [None] * cap                # allocate list with new capacity
        walk = self._front
        for k in range(self._size):              # only consider existing elements
            self._data[k] = old[walk]            # intentionally shift indices
            walk = (walk + 1) % len(old)        # use old size as modulus
        self._front = 0                          # front has been
    def _mostrar(self):
        print(self._data) 
    def __getitem__(self, k):
        if not 0 <= k < self._size:
            raise IndexError('Índice fuera de rango')
        index = (self._front + k) % len(self._data)
        return self._data[index]
        

In [3]:
class NodoBST:
    __slots__ = 'arista', 'izq', 'der'
    
    def __init__(self, arista: 'MediaArista'):
        self.arista = arista
        self.izq = None
        self.der = None

class BST:
    def __init__(self):
        self.raiz = None

    def _evaluar_x(self, arista: 'MediaArista', y_actual: float):
        #En un barrido real, x cambia según la altura y.
        #Calcular la intersección x de la arista a la altura y_actual.
        return arista.origenP.x

    def insertar(self, arista: 'MediaArista', y_actual: float):
        if self.raiz is None:
            self.raiz = NodoBST(arista)
            return
        
        actual = self.raiz
        x_nueva = self._evaluar_x(arista, y_actual)
        
        while True:
            x_actual = self._evaluar_x(actual.arista, y_actual)
            if x_nueva < x_actual:
                if actual.izq is None:
                    actual.izq = NodoBST(arista)
                    break
                actual = actual.izq
            else:
                if actual.der is None:
                    actual.der = NodoBST(arista)
                    break
                actual = actual.der

    def buscarIzquierda(self, x_vertice: float, y_actual: float):
        #Encuentra la arista activa (arista que toca la linea) inmediatamente a la izquierda del vértice actual (v.x)
        actual = self.raiz
        mejor_opcion = None
        
        while actual is not None:
            x_actual = self._evaluar_x(actual.arista, y_actual)
            
            if x_actual <= x_vertice:
                mejor_opcion = actual.arista
                actual = actual.der #Buscar si hay otra arista más cercana por la derecha
            else:
                actual = actual.izq
                
        return mejor_opcion

    def eliminar(self, arista: 'MediaArista'):
        #La eliminación estándar de un BST requiere un método auxiliar recursivo
        self.raiz = self._eliminar_nodo(self.raiz, arista)

    def _eliminar_nodo(self, nodo: Optional['NodoBST'], arista: 'MediaArista') -> Optional['NodoBST']:
        if nodo is None:
            return None
            
        if arista == nodo.arista:
            #Caso 1 y 2: Sin hijos o un solo hijo
            if nodo.izq is None:
                return nodo.der
            if nodo.der is None:
                return nodo.izq
                
            #Caso 3: Dos hijos. Buscar el sucesor (el menor del subárbol derecho)
            sucesor = nodo.der
            while sucesor.izq is not None:
                sucesor = sucesor.izq
            nodo.arista = sucesor.arista
            nodo.der = self._eliminar_nodo(nodo.der, sucesor.arista)
        else:
            nodo.izq = self._eliminar_nodo(nodo.izq, arista)
            nodo.der = self._eliminar_nodo(nodo.der, arista)
            
        return nodo

In [4]:
#Librerias
from enum import Enum, unique

#Clases
class Point:
  verticeID = 0
  def __init__(self, x: float, y: float):
    Point.verticeID += 1
    self.idVertice = Point.verticeID
    self.color = None
    self.x = float(x)
    self.y = float(y)

  def __lt__(self, P): #P son los objetos tipo Point
    if self.y == P.y:
      return self.x < P.x
    return self.y > P.y

  @staticmethod #Permite ejecutar el método directamente desde la clase sin necesidad de instanciar un objeto
  def crossProduct(P1, P2, P3):
    return (P2.x - P1.x) * (P3.y - P1.y) - (P2.y - P1.y) * (P3.x - P1.x)
    #Det positiva: giro izq
    #Det negativa: giro der
    #Det 0: recta

@unique #Todos los valores de la enumeración únicos
class TipoVertice(Enum):
  START = 1
  END = 2
  SPLIT = 3
  MERGE = 4
  REGULAR = 5

class NodoVertice:
  __slots__ = 'P', 'prev', 'next', 'tipoVertice'
  def __init__(self, P, prev, next):
    self.P = P #Objeto Point
    self.tipoVertice = None
    self.prev = prev
    self.next = next

class MediaArista:
  __slots__ = 'origenP', 'twin', 'next', 'prev', 'trianguloPertenece'
  def __init__(self, origenP):
    self.origenP = origenP #De que vértice nace la arista
    self.trianguloPertenece = None
    self.twin = None
    self.next = None
    self.prev = None
    #next -> siguiente media arista
    #prev -> media arista anterior
    
class DCEL:
  def __init__(self, verticesP: list['Point'], mediasAristas: list['MediaArista'], caras: list):
    self.verticesP = verticesP
    self.mediasAristas = mediasAristas
    self.caras = caras
  def masDiagonal(self, P1, P2):
    #Arista dirigida de A -> B y de B -> A
    diagonalAB = MediaArista(P1)
    diagonalBA = MediaArista(P2)
    #Uso de twin para relacionarlas
    diagonalAB.twin = diagonalBA
    diagonalBA.twin = diagonalAB

    #Encontrar medias aristas que salen de cada vértice
    aristaSalidaP1 = None
    aristaSalidaP2 = None

    for arista in self.mediasAristas:
      if arista.origenP == P1:
        aristaSalidaP1 = arista
      elif arista.origenP == P2:
        aristaSalidaP2 = arista

    #Aristas que llegan a los vértices
    aristaEntP1 = aristaSalidaP1.prev
    aristaEntP2 = aristaSalidaP2.prev

    #Conectar diagonal de P1 -> P2
    aristaEntP1.next = diagonalAB
    diagonalAB.prev = aristaEntP1
    diagonalAB.next = aristaSalidaP2
    aristaSalidaP2.prev = diagonalAB

    #Conectar diagonal de P2 -> P1
    aristaEntP2.next = diagonalBA
    diagonalBA.prev = aristaEntP2
    diagonalBA.next = aristaSalidaP1
    aristaSalidaP1.prev = diagonalBA

    #Agregar las nuevas aristas a la lista de la clase
    self.mediasAristas.append(diagonalAB)
    self.mediasAristas.append(diagonalBA)

#Algoritmo principal:
class TrianguladorPoligono:
  def __init__(self, dcel: 'DCEL'): 
    self.dcel = dcel
    self.diagonales = []
    self.triangulos = []

  def clasificarVertices(self):
    clasificacionVertices = {}

    for arista in self.dcel.mediasAristas:
      aristaPrev = arista.prev.origenP
      aristaActual = arista.origenP
      aristaNext = arista.origenP
      determinante = Point.crossProduct(aristaPrev,aristaActual, aristaNext)
      tipo = None
      
      if aristaPrev.y < aristaActual.y and aristaNext.y < aristaActual.y:
        if det > 0:
          tipo = TipoVertice.START 
        else: tipo = TipoVertice.SPILT
      elif aristaPrev.y > aristaActual.y and aristaNext.y > aristaActual.y:
        if det > 0:
          tipo = TipoVertice.END
        else: tipo = TipoVertice.MERGE
      else: tipo = TipoVertice.REGULAR

      clasificacionVertices[aristaActual] = tipo
    return clasificacionVertices

  def hacerMonotono(self):
    clasificacion = self.clasificarVertices()
    
    #Orden de mayor a menor y, en una fila
    verticesLista = sorted(self.dcel.verticesP)
    verticesOrdenados = ArrayQueue()
    for vertice in verticesLista:
      verticesOrdenados.enqueue(vertice)

    #Árbol Binario de Busqueda
    bst = BST()
    helper = {}

    #Línea de barrido
    while not verticesOrdenados.is_empty():
      v = verticesOrdenados.dequeue()
      tipo = clasificacion[v]

      mediaArSale = None
      mediaArLlega = None
      for arista in self.dcel.mediasAristas:
        if arista.origenP == v:
          mediaArSale = arista
        if arista.prev.origenP == v:
          mediaArLlega = arista.prev

      #Casos por cada tipo de vértices
      if tipo == TipoVertice.START:
        bst.insertar(mediaArSale, v.y)
        helper[mediaArSale] = v
      
      elif tipo == TipoVertice.END:
        #Si el helper de la arista que termina es MERGE, se traza una diagonal
        if helper.get(mediaArLlega) and clasificacion.get(helper[mediaArLlega]) == TipoVertice.MERGE:
          self.dcel.masDiagonal(v, helper[mediaArLlega])
        bst.eliminar(mediaArLlega)


      elif tipo == TipoVertice.SPLIT:
        #Busca arista inmediata a la izquierda
        aristaIzq = bst.buscarIquierda(v.x, v.y)
        self.dcel.masDiagonal(v, helper[aristaIzq])
        helper[aristaIzq] = v
        bst.insertar(mediaArSale, v.y)
        helper[mediaArSale] = v

      elif tipo == TipoVertice.MERGE:
        if helper.get(mediaArSale) and clasificacion.get(helper[mediaArSale]) == TipoVertice.MERGE:
          self.dcel.masDiagonal(v, helper[mediaArSale])
        bst.eliminar(mediaArSale)

        aristaIzq = bst.buscarIzquierda(v.x, v.y)
        if aristaIzq and helper.get(aristaIzq) and clasificacion.get(helper[aristaIzq]) == TipoVertice.MERGE:
          self.dcel.masDiagonal(v, helper[aristaIzq])
        if aristaIzq:
          helper[aristaIzq] = v

      elif tipo == TipoVertice.REGULAR:
        if mediaArLlega.origenP > v.y and mediaArSale.next.origenP.y < v.y:
          if helper.get(mediaArLlega) and clasificacion.get(helper[mediaArLlega]) == TipoVertice.MERGE:
            self.dcel.masDiagonal(v, helper[mediaArLlega])
          bst.eliminar(mediaArLlega)
          bst.insertar(mediaArSale, v.y)
          helper[mediaArSale] = v
        else:
          aristaIzq = bst.buscarIzquierda(v.x, v.y)
          if aristaIzq and helper.get(aristaIzq) and clasificacion.get(helper[aristaIzq]) == TipoVertice.MERGE:
            self.dcel.masDiagonal(v, helper[aristaIzq])
          if aristaIzq:
            helper[aristaIzq] = v
    #Retorna objeto ya con poligonos monotonos
    return self.dcel
#Método provisional, se debe de mejorar para apegarse al diseño metodologico
  def triangularPoligonoMonotono(self, yMonotono):
    verticesMonotono = sorted(yMonotono) #De mayor a menor y
    #Se necesitan 3 o más vertices para triangular
    if len(verticesMonotono) < 3:
      return
    pilaVertices = [verticesMonotono[0], verticesMonotono[1]]
    for i in range(2, len(verticesMonotono) - 1):
      verticeActual = verticesMonotono[i]
      ultimoPila = pilaVertices.pop()

      #Triángulos
      self.triangulos.append((pilaVertices[-1], ultimoPila, verticeActual))
      pilaVertices.append(ultimoPila)
      pilaVertices.append(verticeActual)
    return self.triangulos

  def construyeGraficaDual(self):
    arbolDual = {}
    for a, trianguloA in enumerate(self.triangulos):
      arbolDual[a] = []
      for b, trianguloB in enumerate(self.triangulos):
        if a != b:
          #Contar cuantos vértices comparten dos triángulos
          verticesCompartidos = len(set(trianguloA).interseccion(set(trianguloB)))
          if verticesCompartidos == 2:
            arbolDual[a].append(b)
    return arbolDual

  def asignaColores(self, arbolDual: dict):
    colores = {0,1,2}
    for triangulo in self.triangulos:
      if vertice.color is None:
        coloresVecinos = {vecino.color for vecino in triangulo if vecino.color is not None}
        coloresLibres = coloresDisponibles - coloresVecinos
        if coloresLibres:
          vertice.color = list(coloresLibres)[0]

  def seleccionarUbicacion(self, diccPoint):
    conteoColores = {0:0, 1:0, 2:0}
    verticesPorColor = {0:[], 1:[], 2:[]}

    for vertices in self.dcel.verticesP:
      c = vertice.color
      if c in conteoColores:
        conteoColores[c] += 1
        verticesPorColor[c].append(vertice)
    ubicacionesCamaras = min(conteoColores, key=conteoColores.get)
    return verticesPorColor[ubicacionesCamaras]
  
#Agregar para verificar que el objeto Point no exista ya ()
#Agregar función fuera de la clase que llame a triangular poligono


In [ ]:
# ==============================================================================
# ESTRUCTURA DE DATOS: DCEL (Doubly-Connected Edge List)
# ==============================================================================

class Point:
    """Representa una esquina o vértice (x, y) en el plano."""
    def __init__(self, x: float, y: float, index: int = -1):
        self.x = float(x)
        self.y = float(y)
        self.index = index  # Identificador único (v0, v1, v2...)

    def __repr__(self):
        return f"V{self.index}({self.x}, {self.y})"

    def __eq__(self, other):
        return isinstance(other, Point) and self.x == other.x and self.y == other.y


class Face:
    """Representa un triángulo o región interna del polígono."""
    def __init__(self, face_id: int):
        self.face_id = face_id
        self.half_edge = None  # Una semiarista que bordea esta cara

    def __repr__(self):
        return f"Cara_{self.face_id}"


class MediaArista:
    """Representa una semiarista dirigida (Half-Edge)."""
    def __init__(self, origin: Point):
        self.origin = origin          # Punto/Vértice de donde sale la flecha
        self.twin = None              # Flecha gemela en sentido opuesto
        self.next = None              # Siguiente flecha en sentido antihorario
        self.prev = None              # Flecha anterior
        self.incident_face = None     # Cara ubicada a la izquierda de la flecha

    def __repr__(self):
        dest = self.twin.origin if self.twin else "?"
        return f"HalfEdge({self.origin} -> {dest})"


class DCEL:
    """Administra la malla de vértices, medias aristas y caras."""
    def __init__(self):
        self.vertices = []        # Lista de objetos Point
        self.medias_aristas = []  # Lista de objetos MediaArista
        self.caras = []           # Lista de objetos Face
        self.face_count = 0

    def crear_poligono_base(self, coords):
        """Inicializa el polígono simple conectando las paredes en circuito."""
        n = len(coords)
        self.vertices = [Point(x, y, i) for i, (x, y) in enumerate(coords)]

        # Crear la primera cara interna
        f1 = Face(1)
        self.caras.append(f1)

        # Crear semiaristas internas y externas (gemelas)
        edges_in = [MediaArista(v) for v in self.vertices]
        edges_out = [MediaArista(self.vertices[(i + 1) % n]) for i in range(n)]

        for i in range(n):
            e_in = edges_in[i]
            e_out = edges_out[i]

            # 1. Enlazar gemelas (twin)
            e_in.twin = e_out
            e_out.twin = e_in

            # 2. Conectar circuito interno (sentido antihorario)
            e_in.next = edges_in[(i + 1) % n]
            e_in.prev = edges_in[(i - 1) % n]
            e_in.incident_face = f1

            # 3. Conectar circuito externo
            e_out.next = edges_out[(i - 1) % n]
            e_out.prev = edges_out[(i + 1) % n]

            self.medias_aristas.extend([e_in, e_out])

        f1.half_edge = edges_in
        self.face_count = 1

    def masDiagonal(self, v_orig: Point, v_dest: Point):
        """Agrega una diagonal interna entre dos vértices en tiempo constante O(1)."""
        # 1. Buscar las semiaristas que nacen en v_orig y v_dest dentro del recinto
        e_orig = next(e for e in self.medias_aristas if e.origin == v_orig and e.incident_face is not None)
        e_dest = next(e for e in self.medias_aristas if e.origin == v_dest and e.incident_face is not None)

        # 2. Crear las dos nuevas semiaristas gemelas para la diagonal
        d1 = MediaArista(v_orig)
        d2 = MediaArista(v_dest)
        d1.twin = d2
        d2.twin = d1

        # 3. Guardar las referencias anteriores antes de reconectar
        prev_orig = e_orig.prev
        prev_dest = e_dest.prev

        # 4. Reconectar los punteros next y prev para partir la cara en dos
        d1.next = e_dest
        d1.prev = prev_orig
        prev_orig.next = d1
        e_dest.prev = d1

        d2.next = e_orig
        d2.prev = prev_dest
        prev_dest.next = d2
        e_orig.prev = d2

        # 5. Crear una nueva cara y actualizar las pertenencias
        self.face_count += 1
        nueva_cara = Face(self.face_count)
        self.caras.append(nueva_cara)

        # Asignar la cara original al primer circuito
        curr = d1
        while True:
            curr.incident_face = e_orig.incident_face
            curr = curr.next
            if curr == d1:
                break

        # Asignar la nueva cara al segundo circuito
        curr = d2
        while True:
            curr.incident_face = nueva_cara
            curr = curr.next
            if curr == d2:
                break

        d1.incident_face.half_edge = d1
        nueva_cara.half_edge = d2

        self.medias_aristas.extend([d1, d2])
        print(f"✅ Diagonal agregada entre {v_orig} y {v_dest}. Se creó {nueva_cara}.")


# ==============================================================================
# PRUEBA RÁPIDA DE FUNCIONAMIENTO
# ==============================================================================
if __name__ == "__main__":
    dcel = DCEL()
    # Un cuadrado simple de 4 esquinas: (0,0), (2,0), (2,2), (0,2)
    coordenadas = [(0, 0), (2, 0), (2, 2), (0, 2)]
    dcel.crear_poligono_base(coordenadas)

    print(f"Polígono base creado con {len(dcel.vertices)} vértices y {len(dcel.caras)} cara.")

    # Trazamos una diagonal interna de V0 a V2
    dcel.masDiagonal(dcel.vertices, dcel.vertices[1])

    print(f"Total de caras (triángulos) resultantes: {len(dcel.caras)}")